# Accessing ECOSTRESS data

In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
import pydap
import matplotlib.pyplot as plt
# import pydap-specific tools
from pydap.client import get_cmr_urls, open_url
import numpy as np
from pydap.client import to_netcdf as dap_to_netcdf

In [ ]:
auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()

In [ ]:
ECOSTRESS_ccid = "C2076114664-LPCLOUD"
bounding_box = [-128.847656,41.112469,-107.050781,46.679594]
time_range = [dt.datetime(2025, 3, 1), dt.datetime(2025, 3, 14)]


cmr_urls = get_cmr_urls(ccid=ECOSTRESS_ccid, bounding_box = bounding_box, limit=1000, time_range=time_range) # limit by default = 50

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

In [ ]:
%%time
pyds = open_url(cmr_urls[0], protocol="dap4", session=my_session)
pyds.tree()

In [ ]:
keep_vars = ['/SDS/LST',
             "/StandardMetadata/EastBoundingCoordinate", 
             "/StandardMetadata/SouthBoundingCoordinate", 
             "/StandardMetadata/NorthBoundingCoordinate",
             "/StandardMetadata/DayNightFlag",
             "/StandardMetadata/WestBoundingCoordinate",
]

# Stream data into local directory

Stream each remote file into an individual file, since these cannot be aggregated.


In [ ]:
output_path = "data/"

In [ ]:
%%time
dap_to_netcdf(cmr_urls, session=my_session, output_path = output_path, keep_variables=keep_vars)

In [ ]:
dt = xr.open_datatree("data/ECOv002_L2_LSTE_37723_006_20250302T070023_0713_01.nc4")
dt

In [ ]:
dt['SDS/LST'].isel(dim0=slice(0, 1000), dim1=slice(0, 1400)).plot()